Query Enhancement - Query Expansion

In [1]:
#libraries
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap

C:\Users\Dell\AppData\Local\Temp\ipykernel_6664\3228925387.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


loading the dataset and splitting into chunks

In [2]:
loader = TextLoader("langchain_crewai_dataset.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)

putting the chunks into the vectorstore

In [3]:
embedding_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore =  FAISS.from_documents(chunks,embedding_model)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

building the retriever

In [5]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k":5}
)

initializing the LLM and Prompt Setup (Query Enhancement Prompt)

In [6]:
llm = init_chat_model("groq:llama-3.1-8b-instant")
llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000020CF5DA81A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020CF7689160>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

Query expansion chain

In [ ]:
query_prompt = PromptTemplate.from_template("""
You are a helpful assitant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms and useful context.
Original Query: {query}
IMPORTANT: RETURN THE EXPANDED QUERY ONLY, DONT ADD YOUR EXPLANATION TO THE OUTPUT
""")

In [8]:
query_expansion_chain = query_prompt | llm | StrOutputParser()
query_expansion_chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='\nYou are a helpful assitant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms and useful context.\nOriginal Query: {query}\nIMPORTANT: RETURN THE EXPANDED QUERY ONLY, DONT ADD YOUR EXPLANATION TO THE OUTPUT\n')
| ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000020CF5DA81A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020CF7689160>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)
| StrOutputParser()

testing out the query expansion chain

In [13]:
result = query_expansion_chain.invoke({"query":"langchain memory"})
print(result)

langchain memory retrieval 
(langchain OR llms) AND (memory retrieval OR data storage OR caching OR knowledge graph OR memory-based models) 
langchain AND (lens OR chain reaction OR memory model OR knowledge fusion OR memory-augmented models) 
langchain OR (chain reaction OR chain model OR language model chaining OR memory-enhanced models) 
langchain AND (memory-based AI OR knowledge graph database OR semantic memory OR cognitive memory)


Final Rag Prompt

In [14]:
answer_prompt = PromptTemplate.from_template("""
Answer the question based on the context below.
Context:
{context}
Question: {input}
""")

building the document chain

In [15]:
document_chain = create_stuff_documents_chain(llm=llm,prompt=answer_prompt)

final rag pipeline

In [16]:
rag_pipeline = (
    RunnableMap({
        "input": lambda x: x["input"],
        "context": lambda x: retriever.invoke(query_expansion_chain.invoke({"query":x["input"]}))
    })
    | document_chain
)
rag_pipeline

{
  input: RunnableLambda(...),
  context: RunnableLambda(...)
}
| RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
    context: RunnableLambda(format_docs)
  }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
  | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question based on the context below.\nContext:\n{context}\nQuestion: {input}\n')
  | ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000020CF5DA81A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020CF7689160>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('*******

testing out the pipeline

In [18]:
query = {"input":"What types of memory does LangChain support?"}
print(f"Og Query: {query["input"]}")
print("Enhanced Query:")
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("Answer:")
print(response)

Og Query: What types of memory does LangChain support?
Enhanced Query:
{'input': 'What types of memory does LangChain support?', 
 'synonyms': ['types of memory LangChain supports', 'LangChain memory types', 'memory supported by LangChain'], 
 'technical_terms': ['in-memory data structures', 'persistent memory storage', 'data caching', 'memory management', 'LangChain memory framework'], 
 'context': ['LangChain memory module', 'LangChain data storage options', 'LangChain memory architecture', 'memory-intensive tasks in LangChain', 'LangChain memory optimization']}
Answer:
LangChain supports two types of memory modules:

1. ConversationBufferMemory: This allows the LLM to maintain awareness of previous conversation turns.
2. ConversationSummaryMemory: This allows the LLM to summarize long interactions to fit within token limits.


In [20]:
query = {"input":"CrewAI agents?"}
print(f"Og Query: {query["input"]}")
print("Enhanced Query:")
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("Answer:")
print(response)

Og Query: CrewAI agents?
Enhanced Query:
{'input': 'CrewAI agents, crew AI agents, chatbot agents, conversational AI agents, virtual customer service agents, intelligent virtual agents, human-computer interaction, human-machine interface, conversational interfaces, automated customer service, AI-powered agents, machine learning-based agents, natural language processing, NLP, AI assistants, digital workforce, virtual workforce, remote customer service, customer service automation, intelligent customer service, AI-driven customer service'}
Answer:
CrewAI agents have defined roles, such as researcher, planner, or executor, and operate semi-independently within a collaborative context. They form structured workflows and can interact with each other to achieve common goals.
